# Week 39, Day 4: Practice exam

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w039d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload. It doesn't stop you working when time is up..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: P1: Vectors

Write `Vector(x, y)` with `__repr__` (`Vector(1, 2)`), `__eq__` and `__hash__`, `__add__` and `__sub__` with other vectors, `__mul__` and `__rmul__` with numbers (`2 * v` and `v * 2`), `__abs__` (the length, `math.hypot`), and `__bool__` (false only for the zero vector). Return `NotImplemented` for types you don't handle.

In [ ]:
import math

class Vector:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_vectors():\n    \"vectors\"\n    a, b = Vector(1, 2), Vector(3, -1)\n    assert repr(a + b) == \"Vector(4, 1)\" and a - b == Vector(-2, 3) and 2 * a == Vector(2, 4) and a * 3 == Vector(3, 6)\n    assert abs(Vector(3, 4)) == 5 and not Vector(0, 0) and a\n    assert len({Vector(1, 2), Vector(1, 2), b}) == 2 and (a == (1, 2)) is False\n    for bad in [lambda: a + 1, lambda: a * \"x\"]:\n        try:\n            bad()\n            assert False\n        except TypeError:\n            pass")

### Exercise: P2: A stock room

Write `Stock`, a container of item counts. `s[name]` is the count (0 for an item never seen); `s[name] = n` sets it (a negative count raises `ValueError`; 0 removes the item); `len(s)` is the number of items with a count above 0; `name in s` is true when the count is above 0; iterating gives the item names in alphabetical order; and `s.add(other)` returns a **new** `Stock` with the counts of both added, changing neither.

In [ ]:
class Stock:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stock():\n    \"a container\"\n    s = Stock()\n    s[\"tea\"] = 4\n    s[\"rice\"] = 2\n    s[\"milk\"] = 0\n    assert s[\"tea\"] == 4 and s[\"jam\"] == 0 and len(s) == 2 and \"rice\" in s and \"milk\" not in s and \"jam\" not in s\n    assert list(s) == [\"rice\", \"tea\"]\n    s[\"tea\"] = 0\n    assert list(s) == [\"rice\"] and len(s) == 1\n    try:\n        s[\"rice\"] = -1\n        assert False\n    except ValueError:\n        assert s[\"rice\"] == 2\ndef test_add():\n    \"adding stock rooms\"\n    a, b = Stock(), Stock()\n    a[\"tea\"], a[\"rice\"] = 1, 2\n    b[\"tea\"], b[\"jam\"] = 5, 1\n    c = a.add(b)\n    assert [(k, c[k]) for k in c] == [(\"jam\", 1), (\"rice\", 2), (\"tea\", 6)]\n    assert a[\"tea\"] == 1 and b[\"tea\"] == 5 and \"jam\" not in a")

### Exercise: P3: A ticket queue

Write `Priority`, an Enum with `LOW = 1`, `MEDIUM = 2`, `HIGH = 3`; `Ticket`, a frozen dataclass with `ticket_id` (int), `priority` (a `Priority`) and `title`; `TicketError(Exception)`; and `triage(lines)`, which reads lines like `"17|high|Printer on fire"` (id, priority name in any case, title) and returns the tickets sorted with the highest priority first, then by id. Blank lines are skipped. A bad line (wrong number of fields, a bad id, an unknown priority, an empty title) raises `TicketError("line N: 'LINE'")` (N from 1), chained **from** the original error.

In [ ]:
from dataclasses import dataclass
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_triage():\n    \"sorting\"\n    got = triage([\"3|low|Mouse squeaks\", \"\", \"17|HIGH|Printer on fire\", \"4|medium|Slow wifi\", \"2|high|No power\"])\n    assert [t.ticket_id for t in got] == [2, 17, 4, 3] and got[0] == Ticket(2, Priority.HIGH, \"No power\")\n    assert triage([]) == []\ndef test_bad():\n    \"bad lines\"\n    for lines, n in [([\"1|low|ok\", \"x|low|Bad id\"], 2), ([\"5|urgent|Huh\"], 1), ([\"5|low\"], 1), ([\"\", \"6|low|  \"], 2)]:\n        try:\n            triage(lines)\n            assert False, f\"{lines} should raise TicketError\"\n        except TicketError as err:\n            assert str(err) == f\"line {n}: {lines[n - 1]!r}\", f\"got {str(err)!r}\"\n            assert err.__cause__ is not None, \"chain it from the original error\"")

### Exercise: P4: The best discount

Discounts are strategies: anything with a method `apply(total)` returning the new total in pence. Write `Discount`, a `runtime_checkable` Protocol for that method, and three classes: `PercentOff(percent)` (a percent from 1 to 100, else `ValueError`; rounds the result **down** to whole pence), `AmountOff(pence)` (never below 0) and `OverThreshold(threshold, pence)` (takes `pence` off only when the total is **at least** `threshold`). Then `best_price(total, discounts)`: the lowest total any single discount gives, or the total itself if none helps; items that aren't `Discount`s are ignored.

In [ ]:
from typing import Protocol, runtime_checkable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_discounts():\n    \"each discount\"\n    assert PercentOff(10).apply(999) == 899 and PercentOff(100).apply(500) == 0\n    assert AmountOff(300).apply(250) == 0 and AmountOff(100).apply(250) == 150\n    assert OverThreshold(1000, 200).apply(999) == 999 and OverThreshold(1000, 200).apply(1000) == 800\n    for bad in [0, 101]:\n        try:\n            PercentOff(bad)\n            assert False\n        except ValueError:\n            pass\ndef test_best():\n    \"the best\"\n    ds = [PercentOff(10), AmountOff(150), OverThreshold(2000, 400), \"not a discount\"]\n    assert best_price(1000, ds) == 850 and best_price(2000, ds) == 1600 and best_price(3000, ds) == 2600\n    assert best_price(500, []) == 500 and isinstance(AmountOff(1), Discount) and not isinstance(5, Discount)\n    class Free:\n        def apply(self, total):\n            return 0\n    assert best_price(700, [Free()]) == 0, \"any object with apply counts\"")

### Exercise: P5: Test a bounded queue

`BoundedQueue(capacity)` in `bqueue.py` is a first-in, first-out queue that holds at most `capacity` items: `put(item)` raises `OverflowError` when full, `get()` returns the oldest item and raises `LookupError` when empty, and `len(q)` is the number of items. A capacity below 1 raises `ValueError`. Write pytest tests that catch all five broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('bqueue.py', 'w', encoding='utf-8') as f:
    f.write('class BoundedQueue:\n    def __init__(self, capacity):\n        if capacity < 1:\n            raise ValueError("capacity must be at least 1")\n        self.capacity = capacity\n        self._items = []\n\n    def put(self, item):\n        if len(self._items) >= self.capacity:\n            raise OverflowError("queue is full")\n        self._items.append(item)\n\n    def get(self):\n        if not self._items:\n            raise LookupError("queue is empty")\n        return self._items.pop(0)\n\n    def __len__(self):\n        return len(self._items)\n')

In [ ]:
import pytest
from bqueue import BoundedQueue

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"bqueue.py\").read()\nBROKEN = {\n    \"last in, first out\": GOOD.replace(\"self._items.pop(0)\", \"self._items.pop()\"),\n    \"one too many\": GOOD.replace(\"if len(self._items) >= self.capacity:\", \"if len(self._items) > self.capacity:\"),\n    \"capacity 0 allowed\": GOOD.replace(\"if capacity < 1:\", \"if capacity < 0:\"),\n    \"empty gives None\": GOOD.replace('            raise LookupError(\"queue is empty\")', \"            return None\"),\n    \"len counts capacity\": GOOD.replace(\"return len(self._items)\", \"return self.capacity\"),\n}\ndef test_catches():\n    \"your tests pass on the real code and catch all five broken versions\"\n    results = against(source, \"bqueue\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"bqueue.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: C: A car park

Write a small car park system, in layers.

**Domain.** `ParkError(Exception)` is the base, with `Full(ParkError)` and `NotParked(ParkError, LookupError)`. `Stay`, a dataclass with `plate`, `arrived` (minutes since midnight) and `left` (default `None`). Fees are a **strategy**: a function `(minutes) -> pence`. Write `standard_fee`: free for the first 30 minutes, then 150p for each hour or part of an hour after that (so 31 minutes costs 150, 90 minutes costs 150, 91 minutes costs 300), at most 1200p.

**Service.** `CarPark(spaces, fee=standard_fee, clock)`, where `clock` is a function returning the current minute (a keyword argument, required). It keeps the stays, and has:

- `arrive(plate)`: the plate is upper-cased with spaces removed; a car already inside raises `ParkError("AB12CDE is already parked")`; no free space raises `Full("no spaces")`. Records a stay and notifies observers with `("arrived", plate)`.
- `leave(plate)`: the same plate cleaning; a car not inside raises `NotParked("AB12CDE isn't parked")`. Sets `left`, notifies `("left", plate)`, and returns the fee for the stay.
- `free()`: the number of free spaces; `subscribe(listener)` adds an observer.

**Interface.** `handle(line, park)` for lines `in PLATE`, `out PLATE` and `free`, replying `AB12CDE parked`, `AB12CDE paid £1.50` and `3 spaces free`. A `ParkError` becomes `error: ` and its message; anything else typed gets `unknown command`.

In [ ]:
from dataclasses import dataclass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make(spaces=2):\n    now = [600]\n    events = []\n    park = CarPark(spaces, clock=lambda: now[0])\n    park.subscribe(lambda kind, plate: events.append((kind, plate)))\n    return park, now, events\ndef test_fee():\n    \"the fee strategy\"\n    assert [standard_fee(m) for m in [0, 30, 31, 90, 91, 150, 151, 2000]] == [0, 0, 150, 150, 300, 300, 450, 1200]\ndef test_park():\n    \"arriving and leaving\"\n    park, now, events = make()\n    park.arrive(\"ab12 cde\")\n    assert park.free() == 1\n    now[0] = 691\n    assert park.leave(\"AB12CDE\") == 300 and park.free() == 2\n    assert events == [(\"arrived\", \"AB12CDE\"), (\"left\", \"AB12CDE\")]\n    assert park.stays[0] == Stay(\"AB12CDE\", 600, 691)\ndef test_errors():\n    \"errors\"\n    park, now, events = make(1)\n    park.arrive(\"X1\")\n    for call, error, message in [(lambda: park.arrive(\"x1\"), ParkError, \"X1 is already parked\"), (lambda: park.arrive(\"Y2\"), Full, \"no spaces\"),\n                                 (lambda: park.leave(\"Z3\"), NotParked, \"Z3 isn't parked\")]:\n        try:\n            call()\n            assert False, message\n        except error as err:\n            assert str(err) == message, f\"got {str(err)!r}\"\n    assert issubclass(Full, ParkError) and issubclass(NotParked, LookupError)\n    park.leave(\"X1\")\n    park.arrive(\"X1\")\n    assert park.free() == 0, \"a car can come back after leaving\"\ndef test_strategy():\n    \"another fee\"\n    park = CarPark(3, fee=lambda minutes: minutes, clock=lambda: 100)\n    park.arrive(\"A\")\n    assert park.leave(\"A\") == 0\ndef test_handle():\n    \"the interface\"\n    park, now, events = make(1)\n    assert handle(\"in ab12 cde\", park) == \"AB12CDE parked\" and handle(\"free\", park) == \"0 spaces free\"\n    assert handle(\"in Q9\", park) == \"error: no spaces\"\n    now[0] = 660\n    assert handle(\"out AB12CDE\", park) == \"AB12CDE paid \u00a31.50\" and handle(\"out AB12CDE\", park) == \"error: AB12CDE isn't parked\"\n    assert handle(\"park\", park) == \"unknown command\" and handle(\"free\", park) == \"1 spaces free\"")

## How to use this

Treat the first two hours as the real thing: notes closed, no tutor, no web, timer on. Run your code and press **Check** as often as you like. Unlike tomorrow, each problem has a hint and a model solution: **don't open them until the timer has finished**. Then go back through every problem, open the solutions, and compare.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 10 quiz questions | 10 (1 each) |
| B | 5 problems | 70 |
| C | one larger program | 30 |
| | **Total** | **110** |

Count full marks for a problem only when every check passes. 66 (60%) would be a pass, 88 (80%) a distinction.

## Part A: quiz

## Part B: five problems

## Part C: a larger program

## Afterwards

When the timer ends, add up your marks, then go back through each problem with its hint and solution. For every mark you dropped, decide which kind of reason it was:

1. **Didn't know it**: revise the topic from this week's review days.
2. **Misread the question**: the code did something sensible but different. Tomorrow, underline every requirement before you start.
3. **Edge case**: the main case worked; an empty list, a tie or a boundary didn't. Tomorrow, test small and extreme cases before pressing Check.
4. **Ran out of time**: do the problems you're sure of first, and move on after 15 minutes stuck.

Get a good night's sleep. Tomorrow: the Phase 3 exam.